# Lecture: Denoising Diffusion Probabilistic Models (DDPM)

In **C4-1** we studied the *forward* process: a fixed recipe that turns any image
into pure Gaussian noise via the closed form

$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1 - \bar\alpha_t}\,\varepsilon.$$

That process has **no parameters** — it is not a generative model on its own. The
generative model is the **reverse** process: a neural network that learns to undo
the noising, one step at a time. This is the **DDPM** of Ho et al. (2020), and in
this notebook we build, train, and sample from one **entirely from scratch** on
Fashion-MNIST — the same dataset used for the VAE (C2) and GAN (C3), so the
samples are directly comparable.

**The single idea behind training.** Rather than predicting the slightly-cleaner
image $x_{t-1}$ directly, the network predicts the **noise** $\varepsilon$ that
was added to reach $x_t$. The loss is just mean-squared error:

$$\mathcal{L} = \mathbb{E}_{x_0,\, t,\, \varepsilon}\, \big\| \varepsilon - \varepsilon_\theta(x_t, t) \big\|^2$$

That's it — a stable regression objective, no adversarial game, no
reconstruction-vs-KL balancing act. The price is paid at **sampling** time, which
requires running the network once per timestep.

Run the following cell only if you are working with Google Colab to copy the required .py file into the root directory. If you are working locally, ignore this cell.

In [ ]:
!git clone https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/C4-Diffusion_Models/Diffusion.py ./

### Data Preparation

We train on **Fashion-MNIST**, normalised to $[-1, 1]$ to match the noise
distribution of the forward process. This is the same preprocessing used in the
GAN notebook (C3-1), enabling a direct visual comparison of sample quality at the
end.

In [ ]:
import torch
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import FashionMNIST
from Diffusion import DDPM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

TIMESTEPS  = 1000
CHANNELS   = 64
BATCH_SIZE = 128
EPOCHS     = 30
LR         = 2e-4

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_dataset = FashionMNIST(root="./data", train=True, download=True, transform=transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=0, pin_memory=True)

print(f"Training samples: {len(train_dataset)}")

### Model Architecture

The reverse process is parameterised by a **U-Net** $\varepsilon_\theta(x_t, t)$
that takes a noisy 28×28 image and the timestep $t$, and predicts the noise.

Two design choices are characteristic of diffusion models:

- **Timestep conditioning.** The same network must denoise at *every* noise
  level, so it needs to know $t$. We encode $t$ with a **sinusoidal embedding**
  (as in Transformers) and inject it into every residual block.
- **Skip connections.** The U-Net's encoder/decoder skips let fine spatial detail
  bypass the bottleneck — essential for predicting pixel-level noise.

The `DDPM` class in `Diffusion.py` wraps this U-Net together with the fixed
variance schedule and the forward/reverse equations from C4-1.

In [ ]:
model = DDPM(timesteps=TIMESTEPS, channels=CHANNELS).to(device)

# Verify shapes: the U-Net maps (noisy image, timestep) -> predicted noise
_x = torch.randn(4, 1, 28, 28, device=device)
_t = torch.randint(0, TIMESTEPS, (4,), device=device)
_eps = model.model(_x, _t)
print("Predicted-noise shape:", _eps.shape)   # (4, 1, 28, 28)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

### Training

Each training step is remarkably simple (see `DDPM.loss`):

1. Draw a random timestep $t$ for every image in the batch.
2. Sample noise $\varepsilon$ and form $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$ (the closed form from C4-1).
3. Predict $\varepsilon_\theta(x_t, t)$ and minimise $\|\varepsilon - \varepsilon_\theta\|^2$.

Unlike the GAN, the loss **decreases monotonically** and there is only **one**
network and **one** objective — diffusion training is stable. On a Colab T4 GPU,
30 epochs take roughly 10–15 minutes.

> **Tip:** If you are short on time, reduce `EPOCHS` to 10 — samples will be
> rougher but recognisable. Or skip training entirely and load the pre-trained
> weights two cells below.

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=LR)

history = []
for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for x0, _ in train_loader:
        x0 = x0.to(device, non_blocking=True)

        loss = model.loss(x0)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()

    history.append(total / len(train_loader))
    print(f"Epoch {epoch+1:3d}  loss={history[-1]:.4f}")

In [ ]:
# The training loop above ran for 30 epochs; save under a matching name
model.save_model(path="models/ddpm_fashion_mnist_30epochs.pth")

If you do not want to train, load one of the pre-trained models (1000 timesteps, full Fashion-MNIST training set). Two checkpoints are provided: `30epochs` (matches the training cell above) and `50epochs` (longer training, sharper samples).

In [ ]:
model = DDPM(timesteps=TIMESTEPS, channels=CHANNELS).to(device)
#model.load_model(path="models/ddpm_fashion_mnist_50epochs.pth", device=device)            # for running locally
model.load_model(path="AIBIP/C4-Diffusion_Models/models/ddpm_fashion_mnist_50epochs.pth", device=device)  # for running in colab

# Alternatively, the 30-epoch checkpoint:
#model.load_model(path="AIBIP/C4-Diffusion_Models/models/ddpm_fashion_mnist_30epochs.pth", device=device)

### Training Dynamics

In contrast to the oscillating GAN losses (C3-1), the DDPM loss is a plain MSE
regression target and **decreases smoothly**. There is no equilibrium to balance
and no risk of the two-player instabilities seen with GANs — one of the main
practical reasons diffusion models have become dominant.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history, label="Denoising loss (MSE)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("DDPM Training Dynamics — Fashion-MNIST")
ax.legend()
plt.tight_layout()
plt.show()

### Generating Samples

Sampling runs the **reverse process** (`DDPM.sample`): start from pure noise
$x_T \sim \mathcal{N}(0, I)$ and iteratively apply the learned denoising step for
$t = T, T-1, \dots, 1$. Each step removes a little of the predicted noise and adds
back a small amount of fresh noise (ancestral sampling), until a clean image
emerges at $t = 0$.

This loop runs the U-Net **1000 times** for a single batch, so it is noticeably
slower than a GAN's single forward pass — the central trade-off of DDPMs that we
address in later notebooks (DDIM).

In [ ]:
model.eval()

samples = model.sample(16, device=device).cpu()
samples = (samples + 1) / 2        # [-1, 1] -> [0, 1]

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i].squeeze().clamp(0, 1), cmap="gray")
    ax.axis("off")
plt.suptitle("DDPM samples from pure noise — Fashion-MNIST", y=1.02)
plt.tight_layout()
plt.show()

### Watching the Reverse Process

The most instructive view of a diffusion model is the **denoising trajectory**:
the same sample shown at several points along the reverse chain. We start from
unstructured noise and watch a coherent clothing item gradually condense out of
it — the exact mirror image of the forward process from C4-1.

In [ ]:
model.eval()

final, trajectory = model.sample(8, device=device, return_trajectory=True)

# trajectory[k] is the state after k reverse steps; pick a few snapshots.
show_at = [0, 200, 500, 800, 900, 950, 999]
fig, axes = plt.subplots(8, len(show_at), figsize=(13, 12))
for col, k in enumerate(show_at):
    state = (trajectory[k] + 1) / 2
    t_remaining = TIMESTEPS - 1 - k
    for row in range(8):
        axes[row, col].imshow(state[row].squeeze().clamp(0, 1).cpu(), cmap="gray")
        axes[row, col].axis("off")
    axes[0, col].set_title(f"t = {t_remaining}", fontsize=9)
plt.suptitle("Reverse process: pure noise (left) $\\to$ generated image (right)", y=0.91)
plt.tight_layout()
plt.show()

### VAE vs. GAN vs. Diffusion — Where DDPM Fits

| | VAE (C2) | GAN (C3) | Diffusion (C4) |
|---|---|---|---|
| Sample quality | Blurry | Sharp | Sharp |
| Training stability | Stable | Unstable (two-player) | **Stable (single MSE)** |
| Mode coverage | Good | Risk of collapse | **Good (no collapse)** |
| Sampling speed | **Fast** (1 pass) | **Fast** (1 pass) | Slow ($T$ passes) |
| Latent space | Structured | Unstructured | Implicit (noise schedule) |

Diffusion models keep the **sharpness** of GANs and the **stability + mode
coverage** of VAEs. Their one clear weakness — **slow, iterative sampling** — is
exactly what acceleration methods like **DDIM** target, which is the natural next
step from here.

Run the cell below to put GAN and DDPM samples side by side.

In [ ]:
import sys, os

# Make GAN.py importable both locally and on Colab:
# - locally it lives in ../C3-GANs/
# - on Colab it was cloned into AIBIP/C3-GANs/
sys.path.insert(0, os.path.join("..", "C3-GANs"))
sys.path.insert(0, os.path.join("AIBIP", "C3-GANs"))
from GAN import GAN

gan = GAN(latent_dim=64, channels=64).to(device)
#gan.load_model(path="../C3-GANs/models/gan_fashion_mnist.pth", device=device)             # for running locally
gan.load_model(path="AIBIP/C3-GANs/models/gan_fashion_mnist.pth", device=device)           # for running in colab
gan.eval()

n = 8
gan_samples = ((gan.generate(n, device=device) + 1) / 2).cpu()
ddpm_samples = ((model.sample(n, device=device) + 1) / 2).cpu()

fig, axes = plt.subplots(2, n, figsize=(14, 4))
for i in range(n):
    axes[0, i].imshow(gan_samples[i].squeeze().clamp(0, 1), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(ddpm_samples[i].squeeze().clamp(0, 1), cmap="gray")
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("GAN",  fontsize=12)
axes[1, 0].set_ylabel("DDPM", fontsize=12)
plt.suptitle("GAN vs. DDPM — Fashion-MNIST samples", fontsize=12)
plt.tight_layout()
plt.show()

---
## Try It Yourself — Experiment with the DDPM

Work in pairs. **Predict first, then run, then explain in one sentence.**

**A. Read the loss curve.** Compare the DDPM loss curve to the GAN's from C3-1.
Why does this one decrease smoothly while the GAN's oscillated? What single
architectural fact removes the instability? (Hint: how many networks and
objectives are involved?)

**B. Fewer reverse steps, by hand.** In `Diffusion.py`, `DDPM.sample` loops over
all `timesteps`. Make a copy that skips every other step (`range(0, T, 2)` in
reverse) and compare sample quality and runtime. What did you gain, what did you
lose? This is the intuition behind **DDIM** sampling.

**C. The trajectory.** In the reverse-process grid, at roughly which timestep
does the *class* of each item become recognisable? Relate this to C4-1's
observation about where the class first *disappeared* in the forward process —
are the two timesteps similar?

**D. Noise schedule sensitivity.** Re-train (or fine-tune for a few epochs) with
`beta_end = 0.01` instead of `0.02`. Less terminal noise means $x_T$ is *not*
quite pure noise. How does that bias the samples, given that sampling always
starts from $\mathcal{N}(0, I)$?

**E. Diversity, no collapse.** Generate 64 samples and compute the per-pixel
variance exactly as in the GAN mode-collapse cell (C3-1). Diffusion models do not
mode-collapse — verify that the variance is healthily high and the grid is
visibly diverse.